# EMD-Transformer-BiLSTM — Prediksi AQI Jakarta (Kaggle)

Dataset: Jakarta `us_aqi` (Open-Meteo CAMS) + 8 kanal cuaca ERA5.
Pipeline: EMD (12 IMF) → per-IMF Transformer-BiLSTM → Linear fallback → fusi BiLSTM.

**Setelan notebook yang wajib di Kaggle** (sidebar kanan → Session options):
- **Accelerator: GPU T4 x2** — kode memaksa hanya 1 GPU (`CUDA_VISIBLE_DEVICES=0`)
- **Internet: On** — dipakai `git clone` repo (data 8 MB ikut di dalam repo)

Bedanya dengan Colab: tidak ada Google Drive. Semua hasil masuk `/kaggle/working`,
jadi ikut tersimpan sebagai output notebook setelah **Save Version**.
Kalau sesi mati di tengah jalan, jalankan ulang notebook — checkpoint per-IMF
di `results/checkpoints/<lengan>/` membuat training lanjut, bukan mulai dari nol.


In [ ]:
import os

REPO_URL = "https://github.com/madajabbar/riset-aqi-jakarta_ML.git"
BRANCH   = "main"
REPO     = "/kaggle/working/riset-aqi-jakarta_ML"   # /kaggle/working = output notebook

if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch origin {BRANCH} && git -C {REPO} reset --hard origin/{BRANCH}

os.chdir(REPO)
os.makedirs("our_model/results", exist_ok=True)

# Jendela tanggal: uji cepat 1 bulan -> "2022-09-01", "2022-10-01"
START, END = "2022-08-15", "2026-10-01"
print(os.getcwd())

**Pasang dependensi.** Kaggle sudah punya `torch`, `scikit-learn`, `pandas`, `matplotlib`, `numpy` (versi GPU).
Yang kurang cuma `EMD-signal` (paketnya bernama `PyEMD`). Tanpa `-q` biar kalau gagal kelihatan sebabnya.

In [ ]:
%pip install EMD-signal

In [ ]:
import os, sys, torch

aqi, met = "our_data/jakarta_aqi.csv", "our_data/jakarta_met.csv"
assert os.path.exists(aqi), f"{aqi} missing!"
assert os.path.exists(met), f"{met} missing!"
print(f"AQI: {os.path.getsize(aqi):,} bytes | Weather: {os.path.getsize(met):,} bytes")

from PyEMD import EMD          # dari paket EMD-signal
print("PyEMD OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU count:", torch.cuda.device_count())

## Lengan 1 — weather (model kita, RQ2)

`CUDA_VISIBLE_DEVICES=0` dipaksa di sini supaya hanya satu T4 yang dipakai
(dua GPU bikin alokasi memori tidak terduga di pipeline per-IMF).
EMD jalan di CPU (±beberapa menit untuk 36 ribu jam), training di GPU.

In [ ]:
!PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python our_model/train.py --weather --start {START} --end {END} 2>&1 | tee our_model/results/full_weather.log

## Lengan 2 — base (ablasi tanpa cuaca)

Checkpoint per-lengan, jadi lengan ini tidak mewarisi hasil lengan 1.

In [ ]:
!PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 python our_model/train.py --start {START} --end {END} 2>&1 | tee our_model/results/full_base.log

## Hasil + simpan ke output notebook

In [ ]:
from IPython.display import Image, display
import glob, pandas as pd

for f in sorted(glob.glob("our_model/results/*.png")):
    print(f"--- {f} ---"); display(Image(filename=f))
for f in sorted(glob.glob("our_model/results/*.csv")):
    print(f"\n{f}:"); print(pd.read_csv(f, header=None))

In [ ]:
# Ringkas hasil jadi satu arsip kecil di /kaggle/working (ikut Save Version)
import shutil, os

os.makedirs("/kaggle/working/outputs", exist_ok=True)
for f in glob.glob("our_model/results/*.log") + glob.glob("our_model/results/*.csv") + glob.glob("our_model/results/*.png"):
    shutil.copy(f, "/kaggle/working/outputs/")
shutil.make_archive("/kaggle/working/results-kaggle", "zip", "our_model/results")
print(sorted(os.listdir("/kaggle/working/outputs")))